# StoryForge · bản thử riêng 20 luồng v2 cho Anh Đạt

Được tạo từ notebook công khai ghim commit `4da23d040ca5bd6ffc7db75ce2c62be460f55493`.
Payload `settings.py` và `gateway.py` thêm lựa chọn 20; `start_colab.py` xác minh đúng
context sau khi llama.cpp b11981 làm tròn lên bội 256. Model runtime, xác thực, lease,
khóa model và tunnel giữ nguyên. Không tự cập nhật GitHub/main hoặc notebook đang mở.
Ví dụ: yêu cầu context 24000 → thực tế dự kiến 24064 mỗi slot; vẫn dùng 24000 để tính ngân sách.
Sai khác ngoài quy tắc này vẫn bị chặn, không bỏ qua bước xác minh.

1. Tải file này vào Colab bằng **File → Upload notebook**; nên dùng phiên GPU thử riêng.
2. Mặc định Q4_K_M, **20 slot / context 24,000 mỗi slot / output tối đa 20,000**.
   Chạy ô điều khiển và xác nhận mã trong StoryForge như thường lệ.
3. Xem `FORM VỪA NHẬN` trước, sau đó so với bảng **đã áp dụng** khi sẵn sàng.
   Chỉnh số trong chính ô điều khiển này; đặt biến ở một ô khác sẽ bị các phép gán ở đầu ô ghi đè.
4. Chạy ô **Thử 20 request** sau khi máy sẵn sàng. Ô này đi qua gateway thật tại loopback,
   có key và lease thật; không đi qua trình duyệt/Cloudflare/client StoryForge.
5. Để thử đổi cấu hình: đợi test kết thúc, sửa form rồi chạy lại ô điều khiển.
   Nếu vẫn in “không đổi”, gửi lại dòng FORM và bảng đã áp dụng; không gửi key hay mã ghép nối.

Context là tổng token đầu vào + đầu ra; **20000/20000 không hợp lệ** với validator hiện tại.
24,000/20,000 chỉ chừa 4,000 token cho prompt/lịch sử/template.
20 slot yêu cầu pool context **480,000**, bộ nhớ có thể tăng thêm do làm tròn từng slot;
GPU có thể thiếu RAM/VRAM. Chưa benchmark GPU thật.
Giới hạn output không ép model sinh đủ số token. Bấm Dừng ô test sẽ hủy các request do ô tạo.
Chạy lại ô điều khiển cũ sẽ khôi phục mã cũ và lựa chọn 1/3/5; chỉ làm khi không còn request.

## HTML v6
Ô cuối mở trang HTML riêng qua Colab proxy. Dùng máy AI hiện có, không dùng ipywidgets.


In [ ]:
#@title Máy AI cá nhân StoryForge · kết nối bằng mã xác nhận
#@markdown Chọn **Ghép nối / khởi động**, bấm **▶ Chạy**, rồi nhập mã hiện ra vào StoryForge và xác nhận.
thao_tac = "Ghép nối / khởi động" #@param ["Ghép nối / khởi động", "Khôi phục cấu hình trước", "Tạo mã liên kết mới", "Xem trạng thái", "Chẩn đoán", "Dừng máy chủ"]
#@markdown ### Model và thiết bị
model_AI = "Q4_K_M" #@param ["IQ3_M", "IQ4_XS", "Q2_K_P", "Q3_K_M", "Q3_K_P", "Q4_K_M", "Q4_K_P", "Q5_K_M", "Q5_K_P", "Q6_K_P", "Q8_K_P", "GGUF khác"]
thiet_bi = "Tự động" #@param ["Tự động", "GPU", "CPU"]
so_yeu_cau_dong_thoi = 20 #@param [1, 3, 5, 20] {type:"raw"}
#@markdown **BẢN TEST RIÊNG · 20 slot**, hàng đợi tối đa 5. Đổi slot/context/output sẽ nạp lại model khi máy rảnh. Chưa xác minh đủ bộ nhớ trên GPU của anh.
#@markdown ### Ngữ cảnh và độ dài trả lời
#@markdown Context bao gồm cả đầu vào và đầu ra. Output phải ≤ context − 256; 20000/20000 bị từ chối. Pool context = context × số slot. Output chỉ là trần, model có thể kết thúc sớm.
context_tokens = 24000 #@param {type:"integer"}
max_output_tokens = 20000 #@param {type:"integer"}
#@markdown ### GGUF
hf_repo = "HauhauCS/Gemma-4-E4B-Uncensored-HauhauCS-Aggressive" #@param {type:"string"}
gguf_file = "Gemma-4-E4B-Uncensored-HauhauCS-Aggressive-Q8_K_P.gguf" #@param {type:"string"}
hf_revision = "main" #@param {type:"string"}
#@markdown Không cần Colab Secrets. Notebook chỉ nhận credential sau khi người dùng xác nhận mã trong StoryForge.
# File này do build_multiuser_notebook.py sinh; origin không thể đổi trong biểu mẫu.
import base64, io, zipfile, sys, importlib, importlib.metadata, subprocess
SF_ORIGIN = 'https://storyforge-ai-pilot.canhettg113.workers.dev'
if SF_ORIGIN.endswith('.invalid'):
    raise RuntimeError('REVIEW ONLY: origin mẫu .invalid chưa kết nối được tới StoryForge; không gọi mạng.')
try:
    _sf_httpx_version = importlib.metadata.version('httpx')
except importlib.metadata.PackageNotFoundError:
    _sf_httpx_version = None
if _sf_httpx_version != '0.28.1':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'httpx==0.28.1'],
                   check=True, timeout=180)
from pathlib import Path
_sf_folder = Path('/content/storyforge-personal-ai/source')
_sf_folder.mkdir(parents=True, exist_ok=True)
_sf_payload = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_sf_payload))) as _sf_zip:
    _sf_zip.extractall(_sf_folder)
if str(_sf_folder) not in sys.path:
    sys.path.insert(0, str(_sf_folder))
import runtime, connection, settings, linked_connection, start_colab
for _sf_module in (runtime, connection, settings, linked_connection, start_colab):
    importlib.reload(_sf_module)
_sf_personal_state = globals().get('SF_STATE', {})
if isinstance(_sf_personal_state, dict) and any(
        _sf_personal_state.get(_sf_name) is not None
        and _sf_personal_state[_sf_name].poll() is None
        for _sf_name in ('model', 'gateway', 'tunnel')):
    raise RuntimeError('Dừng máy chủ notebook cá nhân trong runtime này trước khi ghép nối nhiều người dùng.')
start_colab.ROOT = Path('/content/storyforge-personal-ai')
SF_LINKED_STATE = globals().get('SF_LINKED_STATE', {})
SF_LINKED_STATE['multiuser'] = True
_sf_settings = {'model_choice': model_AI, 'device': thiet_bi,
    'parallel': so_yeu_cau_dong_thoi,
    'context': context_tokens, 'max_tokens': max_output_tokens,
    'repo': hf_repo, 'filename': gguf_file, 'revision': hf_revision}

def _sf_wait_message(status):
    labels = {'pending': 'Đang chờ Anh/Chị xác nhận trong StoryForge…',
              'approved': 'Đã xác nhận. StoryForge đang chuẩn bị kết nối…',
              'provisioning': 'Đang cấp đường kết nối riêng…',
              'slow_down': 'Máy chủ yêu cầu chờ lâu hơn trước lần kiểm tra tiếp theo…'}
    if status in labels:
        print(labels[status], flush=True)

try:
    print('BẢN TEST RIÊNG 20 SLOT v2 · sửa kiểm tra context làm tròn · nền 4da23d0')
    if thao_tac == 'Ghép nối / khởi động':
        print('FORM VỪA NHẬN:', 'model=', model_AI, 'context=', context_tokens,
              'output=', max_output_tokens, 'slot=', so_yeu_cau_dong_thoi, flush=True)
        _sf_running = SF_LINKED_STATE.get('applied_config', {})
        print('ĐÃ ÁP DỤNG TRƯỚC:', {key: _sf_running.get(key)
              for key in ('context', 'max_tokens', 'parallel')}, flush=True)
        # Check input BEFORE creating/polling a pairing or changing the model.
        _sf_validated = settings.resolve_multiuser_settings(_sf_settings, has_gpu=True)
    if thao_tac == 'Xem trạng thái':
        start_colab.show_linked_panel(SF_LINKED_STATE)
    elif thao_tac == 'Chẩn đoán':
        start_colab.diagnostics(SF_LINKED_STATE)
    elif thao_tac == 'Dừng máy chủ':
        start_colab.stop(SF_LINKED_STATE)
        print('Đã dừng các tiến trình do notebook quản lý. Kết nối sẽ hết lease nếu không còn heartbeat.')
    else:
        if thao_tac == 'Khôi phục cấu hình trước':
            _sf_previous = SF_LINKED_STATE.get('previous_model_config')
            if not _sf_previous:
                raise ValueError('Chưa có cấu hình trước để khôi phục.')
            _sf_settings = {'model_choice': 'GGUF khác', 'device': 'GPU' if _sf_previous['device'] == 'gpu' else 'CPU',
                'parallel': _sf_previous.get('parallel', 1),
                **{key: _sf_previous[key] for key in ('context', 'max_tokens', 'repo', 'filename', 'revision')}}
        _sf_link = linked_connection.LinkedConnection(SF_ORIGIN, SF_LINKED_STATE)
        if thao_tac == 'Tạo mã liên kết mới':
            start_colab.prepare_new_link(SF_LINKED_STATE)
            _sf_pair = _sf_link.new_pair(label='Colab AI')
        else:
            _sf_pair = _sf_link.ensure_pair(label='Colab AI')
        if _sf_pair:
            print('Mã liên kết dùng một lần:', _sf_pair['code'], flush=True)
            print('Nhập mã này trong Cài đặt kết nối máy AI của StoryForge và xác nhận rõ ràng.', flush=True)
        if SF_LINKED_STATE.get('linked_credentials'):
            _sf_credentials = SF_LINKED_STATE['linked_credentials']
        else:
            print('Đang chờ xác nhận; lần chạy lại sẽ tiếp tục cùng mã liên kết.', flush=True)
            _sf_credentials = _sf_link.poll_until_claim(max_wait=600, on_status=_sf_wait_message)
        print('Đã nhận quyền phiên trong bộ nhớ. Credential không được in ra notebook.', flush=True)
        start_colab.start_linked(SF_LINKED_STATE, _sf_credentials, SF_ORIGIN, settings=_sf_settings)
        start_colab.show_linked_panel(SF_LINKED_STATE)
except KeyboardInterrupt:
    print('Đã dừng lượt khởi động/tải model. Chạy lại ô để thử tiếp; tải lại trang không khởi động lại máy Colab.', flush=True)
    start_colab.show_linked_panel(SF_LINKED_STATE)
except Exception as _sf_error:
    if ((isinstance(_sf_error, (ValueError, TimeoutError)) and type(_sf_error).__name__ != 'JSONDecodeError')
            or isinstance(_sf_error, runtime.RuntimeActionError)):
        print('Chưa áp dụng yêu cầu:', str(_sf_error))
    else:
        print('CHƯA SẴN SÀNG:', type(_sf_error).__name__)
        print('Chi tiết thô và nhật ký riêng tư được ẩn. Chọn Chẩn đoán để xem trạng thái tiến trình.')
    start_colab.show_linked_panel(SF_LINKED_STATE)


In [ ]:
#@title Mở trang HTML test dịch 20 luồng
import asyncio
import hashlib
import json
import math
import re
import time
from pathlib import Path
import httpx

async def consume_stream(response, row, writer=None, on_update=None):
    """Require an SSE terminator; EOF alone is not a successful inference."""
    row['http'] = response.status_code
    row['request_id'] = response.headers.get('x-request-id')
    if response.status_code != 200:
        row['error'] = 'HTTP_' + str(response.status_code)
        # Do not print raw server responses or credentials.
        await response.aread()
        return
    done = False
    async for line in response.aiter_lines():
        if not line.startswith('data:'):
            continue
        data = line[5:].strip()
        if data == '[DONE]':
            done = True
            break
        event = json.loads(data)
        if event.get('error'):
            row['error'] = 'SSE_ERROR'
            return
        if event.get('usage'):
            row['usage'] = event['usage']
        for choice in event.get('choices', []):
            content = choice.get('delta', {}).get('content') or ''
            if content:
                if row['first_token_seconds'] is None:
                    row['first_token_seconds'] = round(time.perf_counter() - row['started'], 3)
                row['characters'] += len(content)
                row['preview'] = (row['preview'] + content)[:120]
                if writer is not None:
                    writer.write(content)
                if on_update is not None:
                    on_update(content)
            if choice.get('finish_reason'):
                row['finish_reason'] = choice['finish_reason']
    if not done:
        row['error'] = 'STREAM_WITHOUT_DONE'

async def measure_prompt_tokens(client, messages):
    response = await client.post('/apply-template', json={'messages': messages,
        'add_generation_prompt': True, 'chat_template_kwargs': {'enable_thinking': False}})
    response.raise_for_status()
    prompt = response.json()['prompt']
    if not isinstance(prompt, str):
        raise ValueError('Không đọc được chat template của model; chưa gửi dịch.')
    response = await client.post('/tokenize', json={'content': prompt, 'add_special': True,
                                                  'parse_special': True})
    response.raise_for_status()
    tokens = response.json()['tokens']
    if not isinstance(tokens, list) or not all(type(token) is int for token in tokens):
        raise ValueError('Không đọc được tokenizer của model; chưa gửi dịch.')
    return len(tokens)

# Match JS UTF-16 indices and JS whitespace from source-reader.js exactly.
_JS_SPACE = r'[\t\n\v\f\r \u00a0\u1680\u2000-\u200a\u2028\u2029\u202f\u205f\u3000\ufeff]'
_BOUNDARIES = [re.compile(pattern) for pattern in (
    r'\n' + _JS_SPACE + r'*\n+', r'\n+',
    r'''[.!?。！？…]+["'”’)\]]?''' + _JS_SPACE + '+', _JS_SPACE + '+')]


def _units(text):
    raw = text.encode('utf-16-le')
    return ''.join(chr(raw[i] | raw[i + 1] << 8) for i in range(0, len(raw), 2))


def _from_units(text):
    return text.encode('utf-16-le', errors='surrogatepass').decode('utf-16-le')


def _cut(source, target, start=0):
    remaining = len(source) - start
    if remaining <= target:
        return remaining
    minimum = max(1, math.floor(target * .6))
    maximum = min(remaining, math.ceil(target * 1.25))
    for boundary in _BOUNDARIES:
        before = -1
        after = -1
        for match in boundary.finditer(source, start):
            end = match.end() - start
            if minimum <= end <= target:
                before = end
            if target < end <= maximum:
                after = end
                break
            if end > maximum:
                break
        if before > 0:
            return before
        if after > 0:
            return after
    if 0xD800 <= ord(source[start + target - 1]) <= 0xDBFF and 0xDC00 <= ord(source[start + target]) <= 0xDFFF:
        return target + 1
    return target


def split_story_text(text, size):
    """Large-file web boundary policy; preserve all text, including the last tail."""
    if type(size) is not int or size < 1:
        raise ValueError('Kích thước chunk phải là số nguyên dương.')
    source = _units(text)
    chunks = []
    offset = 0
    while offset < len(source):
        cut = _cut(source, size, offset)
        chunks.append(_from_units(source[offset:offset + cut]))
        offset += cut
    return chunks


def translation_messages(source, prompt=''):
    # Same auto-language user directive and source marker as request-contract.js.
    user = '''[YÊU CẦU NGÔN NGỮ NGUỒN]
- Tự động phát hiện ngôn ngữ nguồn của đoạn văn bên dưới.
- Dịch trực tiếp toàn bộ nội dung sang tiếng Việt mượt mà, tự nhiên.
- Nếu nguồn đã là tiếng Việt convert hoặc dịch máy, biên tập lại thành tiếng Việt tự nhiên.
- Chỉ trả về bản tiếng Việt cuối cùng; không giải thích và không bỏ đoạn.

[Đoạn nguồn]
''' + source.strip()
    return ([{'role': 'system', 'content': prompt.strip()}] if prompt.strip() else []) + [
        {'role': 'user', 'content': user}]


async def plan_story_chunks(text, size, config, output, client, prompt='', on_progress=None):
    if not text.strip():
        raise ValueError('Anh Đạt, file truyện rỗng.')
    if type(output) is not int or not 1 <= output <= config['max_tokens']:
        raise ValueError('Output/chunk phải từ 1 đến trần output đã áp dụng của máy.')
    pending = list(reversed(split_story_text(text, size)))
    plans = []
    while pending:
        source = pending.pop()
        messages = translation_messages(source, prompt)
        if source.strip():
            measured = await measure_prompt_tokens(client, messages)
        else:
            measured = 0
        if measured + output + 256 > config['context']:
            units = _units(source)
            cut = _cut(units, max(1, len(units) // 2))
            if cut >= len(units):
                raise ValueError('Prompt và ngân sách output không vừa context, ngay cả chunk tối thiểu. Giảm output hoặc tăng context.')
            pending.extend([_from_units(units[cut:]), _from_units(units[:cut])])
            continue
        plans.append({'source': source, 'messages': messages, 'prompt_tokens': measured, 'output_tokens': output})
        if on_progress:
            on_progress(len(plans), len(pending))
        elif len(plans) % 20 == 0:
            print('Anh Đạt · đã đo', len(plans), 'chunk; còn', len(pending), 'đoạn đang chờ kiểm tra.', flush=True)
    if ''.join(plan['source'] for plan in plans) != text:
        raise ValueError('Chia chunk làm sai nội dung nguồn; đã dừng trước khi dịch.')
    return plans


def _write_json(path, value):
    temporary = path.with_suffix('.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(path)


async def translate_story_chunks(state, plans, workers, folder, *, retry_failed=False, transport=None,
                                 on_event=None, quiet=False):
    config = state.get('applied_config', {})
    if not state.get('ready') or not state.get('key'):
        raise ValueError('Anh Đạt, chờ ô máy AI nạp xong trước.')
    if type(workers) is not int or not 1 <= workers <= min(20, config.get('parallel', 0)):
        raise ValueError('Số luồng dịch phải từ 1 đến số slot đã áp dụng, tối đa 20.')
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    fingerprint = hashlib.sha256(json.dumps(plans, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
    checkpoint = folder / 'checkpoint.json'
    saved = json.loads(checkpoint.read_text(encoding='utf-8')) if checkpoint.exists() else {}
    if saved and saved.get('fingerprint') != fingerprint:
        raise ValueError('Thư mục kết quả thuộc cách chia/prompt khác; chọn thư mục mới.')
    rows = saved.get('chunks') or [{'id': i + 1, 'status': 'pending'} for i in range(len(plans))]
    for row in rows:
        if row['status'] == 'done' and not (folder / f"chunk-{row['id']:05d}.txt").is_file():
            row['status'] = 'pending'
    pending = iter(i for i, row in enumerate(rows) if row['status'] in ('pending', 'running', 'cancelled')
                   or (retry_failed and row['status'] in ('failed', 'truncated')))
    peak = {'processing_slots': 0, 'dispatched_or_streaming': 0, 'queued': 0, 'status_errors': 0}
    tasks = []

    def emit(kind, **values):
        if on_event is not None:
            on_event(kind, **values)

    def persist():
        _write_json(checkpoint, {'fingerprint': fingerprint, 'chunks': rows})

    async with httpx.AsyncClient(base_url='http://127.0.0.1:18081',
            headers={'Authorization': 'Bearer ' + state['key']}, follow_redirects=False, trust_env=False,
            timeout=httpx.Timeout(60, connect=5), limits=httpx.Limits(max_connections=25), transport=transport) as client:

        async def sample():
            response = await client.get('/v1/runtime', timeout=5)
            response.raise_for_status()
            value = response.json()
            for key, count in (
                ('processing_slots', sum(s.get('is_processing') is True for s in value['slots'])),
                ('dispatched_or_streaming', sum(r['state'] in ('dispatched', 'streaming') for r in value['requests'])),
                ('queued', sum(r['state'] == 'queued' for r in value['requests']))):
                peak[key] = max(peak[key], count)
            return value

        initial = await sample()
        if initial['requests'] or any(s.get('is_processing') for s in initial['slots']):
            raise ValueError('Anh Đạt, máy còn request khác; chờ hoàn tất trước khi dịch chunk.')
        if initial['max_concurrent'] != config['parallel'] or len(initial['slots']) != config['parallel']:
            raise ValueError('Slot gateway/model chưa khớp cấu hình; chưa gửi dịch.')
        emit('job', plans=plans, rows=rows, folder=str(folder), workers=workers)

        async def worker(lane):
            for index in pending:
                plan, row = plans[index], rows[index]
                if not plan['source'].strip():
                    (folder / f"chunk-{row['id']:05d}.txt").write_text(plan['source'], encoding='utf-8')
                    row['status'] = 'done'
                    persist()
                    continue
                row.update(status='running', http=None, error=None, characters=0, preview='',
                           finish_reason=None, first_token_seconds=None, started=time.perf_counter())
                row['input_characters'] = len(_units(plan['source']))
                row.pop('usage', None)
                emit('start', lane=lane, index=index, row=row, plan=plan)
                try:
                    with (folder / f"chunk-{row['id']:05d}.txt").open('w', encoding='utf-8') as writer:
                        async with asyncio.timeout(1800):
                            async with client.stream('POST', '/v1/chat/completions', json={
                                'model': config.get('alias', 'storyforge-local'), 'messages': plan['messages'],
                                'max_tokens': plan['output_tokens'], 'stream': True,
                                'chat_template_kwargs': {'enable_thinking': False},
                                'stream_options': {'include_usage': True}}) as response:
                                row['http'] = response.status_code
                                emit('response', lane=lane, index=index, row=row)
                                await consume_stream(response, row, writer,
                                    on_update=lambda text: emit('token', lane=lane, index=index, row=row, text=text))
                    if not row['error'] and not row['characters']:
                        row['error'] = 'EMPTY_TRANSLATION'
                    row['status'] = 'failed' if row['error'] else (
                        'truncated' if row['finish_reason'] == 'length' else 'done')
                except asyncio.CancelledError:
                    row['status'] = 'cancelled'
                    raise
                except Exception as error:
                    row.update(status='failed', error=type(error).__name__)
                finally:
                    row['total_seconds'] = round(time.perf_counter() - row.pop('started'), 3)
                    row['output_input_ratio'] = round(row['characters'] / row['input_characters'], 3)
                    persist()
                    emit('finish', lane=lane, index=index, row=row)
                    if not quiet:
                        print('Anh Đạt · chunk', row['id'], '/', len(plans), '·', row['status'],
                              '· HTTP', row['http'], '·', row['total_seconds'], 'giây', flush=True)

        tasks = [asyncio.create_task(worker(lane)) for lane in range(min(workers, len(plans)))]
        try:
            next_report = 0
            while any(not task.done() for task in tasks):
                try:
                    observed = await sample()
                    emit('runtime', value=observed, peak=peak)
                except (httpx.HTTPError, ValueError, KeyError, TypeError):
                    peak['status_errors'] += 1
                if not quiet and time.monotonic() >= next_report:
                    print('Anh Đạt · đã dịch', sum(r['status'] == 'done' for r in rows), '/', len(rows),
                          '· peak slot', peak['processing_slots'], flush=True)
                    next_report = time.monotonic() + 10
                await asyncio.wait(tasks, timeout=.5)
            await asyncio.gather(*tasks)
        finally:
            for task in tasks:
                if not task.done():
                    task.cancel()
            await asyncio.gather(*tasks, return_exceptions=True)
            persist()
            merged = folder / 'BAN-DICH.txt'
            with merged.open('w', encoding='utf-8') as writer:
                for i, row in enumerate(rows):
                    path = folder / f"chunk-{row['id']:05d}.txt"
                    if i:
                        writer.write('\n\n')
                    if row['status'] != 'done':
                        writer.write(f"[CHUNK {row['id']} · {row['status']} · CẦN KIỂM TRA]\n")
                    if path.exists():
                        writer.write(path.read_text(encoding='utf-8'))
            emit('closed', rows=rows, merged_file=str(merged))
    report = {'total': len(rows), 'completed': sum(r['status'] == 'done' for r in rows),
              'peak': peak, 'merged_file': str(merged), 'chunks': rows}
    _write_json(folder / 'report.json', report)
    if not quiet:
        print('Anh Đạt · Kết quả:', report['completed'], '/', report['total'], '·', merged, flush=True)
    return report


async def run_colab_story_chunks(state, source_path='', *, upload_new=False, size=6000, workers=20,
                                output=20000, prompt='', preview=True, retry_failed=False,
                                on_event=None, quiet=False):
    if globals().get('SF_CHUNK_TEST_BUSY'):
        raise ValueError('Anh Đạt, ô dịch chunk trước vẫn đang chạy.')
    if not state.get('ready') or not state.get('key'):
        raise ValueError('Anh Đạt, chạy ô máy AI tới khi nạp xong trước.')
    if type(size) is not int or not 500 <= size <= 8000:
        raise ValueError('Chunk ban đầu phải từ 500 đến 8000 ký tự, giống ô chỉnh trên web.')
    if type(workers) is not int or not 1 <= workers <= min(20, state.get('applied_config', {}).get('parallel', 0)):
        raise ValueError('Luồng dịch phải từ 1 đến số slot máy đã nạp, tối đa 20. Không tự nạp lại model.')
    source = Path(source_path) if source_path.strip() else globals().get('SF_CHUNK_SOURCE_FILE')
    if upload_new or source is None or not source.is_file():
        from google.colab import files
        print('Anh Đạt, chọn một file truyện TXT UTF-8.', flush=True)
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError('Chọn đúng một file truyện cho mỗi lượt thử.')
        source = Path('/content/storyforge-chunk-input.txt')
        source.write_bytes(next(iter(uploaded.values())))
    globals()['SF_CHUNK_SOURCE_FILE'] = source
    text = source.read_bytes().decode('utf-8-sig')
    signature = hashlib.sha256(json.dumps({'source': text, 'size': size, 'output': output,
        'prompt': prompt, 'config': state['applied_config']}, ensure_ascii=False, sort_keys=True).encode()).hexdigest()[:16]
    folder = Path('/content/storyforge-chunk-test') / signature
    globals()['SF_CHUNK_TEST_BUSY'] = True
    try:
        if not quiet:
            print('Anh Đạt · đang đo token và chia chunk; chưa gọi inference.', flush=True)
        progress = (lambda done, pending: on_event('planning', done=done, pending=pending)) if on_event else None
        async with httpx.AsyncClient(base_url='http://127.0.0.1:18080', timeout=30,
                follow_redirects=False, trust_env=False) as client:
            plans = await plan_story_chunks(text, size, state['applied_config'], output, client, prompt, progress)
        folder.mkdir(parents=True, exist_ok=True)
        _write_json(folder / 'chunk-plan.json', [{'id': i + 1, 'source': p['source'],
            'source_utf16_chars': len(_units(p['source'])), 'prompt_tokens': p['prompt_tokens'],
            'output_tokens': p['output_tokens']} for i, p in enumerate(plans)])
        if on_event:
            on_event('plan', plans=plans, folder=str(folder), workers=workers)
        if not quiet:
            print('Anh Đạt ·', len(plans), 'chunk; tối đa', min(workers, len(plans)), 'luồng. Kế hoạch:', folder / 'chunk-plan.json')
            for i, plan in enumerate(plans[:20]):
                print('Chunk', i + 1, '·', len(_units(plan['source'])), 'ký tự · prompt', plan['prompt_tokens'],
                      'token · output tối đa', plan['output_tokens'])
        if preview:
            if not quiet:
                print('Anh Đạt, bỏ chọn chi_xem_chunks rồi chạy lại ô này để dịch; giữ tai_file_moi=False.')
            return {'preview': True, 'chunks': len(plans), 'folder': str(folder)}
        return await translate_story_chunks(state, plans, workers, folder, retry_failed=retry_failed,
                                             on_event=on_event, quiet=quiet)
    finally:
        globals()['SF_CHUNK_TEST_BUSY'] = False


"""HTML test UI on loopback HTTP, independent of notebook widget comms."""
import asyncio
import json
import secrets
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path


class StoryHtmlApp:
    def __init__(self, state, runner, page, folder='/content/storyforge-html-test'):
        self.state, self.runner, self.page = state, runner, page
        self.nonce = secrets.token_urlsafe(24)
        self.lock = threading.RLock()
        self.source = Path(folder) / 'input.txt'
        self.source.parent.mkdir(parents=True, exist_ok=True)
        self.thread = self.loop = self.task = None
        self.active = self.stopping = False
        self.phase, self.error, self.started = 'idle', '', None
        self.finished = None
        self.plans, self.rows, self.lanes, self.buffers, self.tails = [], [], {}, {}, {}
        self.measured = self.pending = 0
        self.runtime, self.peak, self.merged = {}, {}, None
        config = state.get('applied_config', {})
        self.options = {'size': 6000, 'workers': min(20, config.get('parallel', 1)),
            'output': min(20000, config.get('max_tokens', 20000)),
            'prompt': 'Giữ nhất quán tên riêng, xưng hô và sắc thái truyện. Không tóm tắt hoặc bỏ đoạn.',
            'preview': False, 'retry_failed': False}

    def event(self, kind, **value):
        with self.lock:
            if kind == 'planning':
                self.measured, self.pending = value['done'], value['pending']
            elif kind in ('plan', 'job'):
                self.plans = value['plans']
                self.rows = value.get('rows') or [{'id': i + 1, 'status': 'pending'} for i in range(len(self.plans))]
                self.folder = Path(value['folder'])
                if kind == 'job' and not self.stopping:
                    self.phase = 'running'
            elif kind == 'start':
                index = value['index']
                self.lanes[value['lane']] = index
                self.buffers[index], self.tails[index] = [], ''
            elif kind == 'token':
                index, text = value['index'], value['text']
                self.buffers.setdefault(index, []).append(text)
                self.tails[index] = (self.tails.get(index, '') + text)[-340:]
            elif kind == 'finish':
                self.buffers.pop(value['index'], None)
            elif kind == 'runtime':
                self.runtime, self.peak = value['value'], dict(value['peak'])
            elif kind == 'closed':
                self.merged = value['merged_file']

    def start(self, options):
        with self.lock:
            if self.active:
                raise ValueError('Lượt trước còn chạy. Bấm Dừng và đợi trước khi chạy tiếp.')
            if not self.source.is_file():
                raise ValueError('Anh Đạt, chọn một file TXT trước.')
            current = dict(self.options)
            current.update({k: v for k, v in options.items() if k in current})
            config = self.state.get('applied_config', {})
            for key, low, high in (('size', 500, 8000), ('workers', 1, min(20, config.get('parallel', 1))),
                                   ('output', 1, config.get('max_tokens', 20000))):
                if type(current[key]) is not int or not low <= current[key] <= high:
                    raise ValueError(f'{key} phải từ {low} đến {high}.')
            if not isinstance(current['prompt'], str) or len(current['prompt']) > 32000:
                raise ValueError('Prompt không hợp lệ hoặc dài quá 32.000 ký tự.')
            self.options = current
            self.active, self.stopping = True, False
            self.phase, self.error, self.started = 'planning', '', time.monotonic()
            self.finished = None
            self.plans, self.rows, self.lanes, self.buffers, self.tails = [], [], {}, {}, {}
            self.runtime, self.peak, self.merged = {}, {}, None
            self.measured = self.pending = 0
            self.thread = threading.Thread(target=self._work, daemon=True, name='StoryForge-HTML-translation')
            self.thread.start()

    def _work(self):
        async def run():
            with self.lock:
                self.loop = asyncio.get_running_loop()
                self.task = asyncio.create_task(self.runner(self.state, str(self.source),
                    **self.options, on_event=self.event, quiet=True))
                if self.stopping:
                    self.task.cancel()
            return await self.task
        try:
            result = asyncio.run(run())
            with self.lock:
                self.phase = 'preview' if self.options['preview'] else 'done'
                if result and result.get('merged_file'):
                    self.merged = result['merged_file']
        except asyncio.CancelledError:
            with self.lock:
                self.phase = 'stopped'
        except Exception as error:
            with self.lock:
                self.phase = 'error'
                self.error = str(error) if isinstance(error, (ValueError, UnicodeError)) else type(error).__name__
        finally:
            with self.lock:
                self.active, self.loop, self.task = False, None, None
                self.finished = time.monotonic()

    def stop(self):
        with self.lock:
            if self.active:
                self.stopping, self.phase = True, 'stopping'
                if self.loop and self.task:
                    self.loop.call_soon_threadsafe(self.task.cancel)

    def snapshot(self):
        with self.lock:
            rows = [{key: r.get(key) for key in ('id', 'status', 'characters', 'error', 'http', 'first_token_seconds')}
                    for r in self.rows]
            lanes = []
            for lane in range(20):
                index = self.lanes.get(lane)
                row = dict(self.rows[index]) if index is not None else {}
                plan = self.plans[index] if index is not None else {}
                lanes.append({'lane': lane, 'index': index, 'status': row.get('status', 'pending'),
                    'characters': row.get('characters', 0), 'first': row.get('first_token_seconds'),
                    'input': len(plan.get('source', '')), 'prompt_tokens': plan.get('prompt_tokens', 0),
                    'text': self.tails.get(index, ''), 'error': row.get('error'),
                    'seconds': round(time.perf_counter() - row['started']) if 'started' in row else row.get('total_seconds', 0)})
            config = self.state.get('applied_config', {})
            return {'phase': self.phase, 'active': self.active, 'error': self.error,
                'seconds': round((self.finished if self.finished is not None else time.monotonic()) - self.started) if self.started else 0,
                'measured': self.measured, 'pending': self.pending, 'total': len(rows),
                'completed': sum(r['status'] == 'done' for r in rows),
                'processing': sum(s.get('is_processing') is True for s in self.runtime.get('slots', [])),
                'peak': self.peak.get('processing_slots', 0), 'lanes': lanes, 'chunks': rows,
                'has_file': self.source.is_file(), 'download': bool(self.merged and not self.active),
                'options': dict(self.options), 'config': {k: config.get(k) for k in ('parallel', 'context', 'max_tokens')}}

    def chunk(self, index):
        with self.lock:
            if not 0 <= index < len(self.plans):
                raise ValueError('Chunk chưa có trong kế hoạch.')
            source = self.plans[index]['source']
            text = ''.join(self.buffers[index]) if index in self.buffers else None
            path = self.folder / f'chunk-{index + 1:05d}.txt'
        if text is None:
            text = path.read_text(encoding='utf-8') if path.is_file() else ''
        return {'source': source, 'output': text}

    def serve(self, port=0):
        app = self
        class Handler(BaseHTTPRequestHandler):
            def log_message(self, *args):
                pass

            def send(self, status, data, content_type='application/json; charset=utf-8'):
                body = data if isinstance(data, bytes) else json.dumps(data, ensure_ascii=False).encode()
                self.send_response(status)
                self.send_header('Content-Type', content_type)
                self.send_header('Content-Length', str(len(body)))
                self.send_header('Cache-Control', 'no-store')
                self.send_header('X-Content-Type-Options', 'nosniff')
                self.end_headers()
                self.wfile.write(body)

            def do_GET(self):
                if self.path.split('?')[0] in ('/', '/index.html'):
                    self.send(200, app.page.replace('__SF_NONCE__', app.nonce).encode(), 'text/html; charset=utf-8')
                    return
                self.dispatch(False)

            def do_POST(self):
                self.dispatch(True)

            def dispatch(self, post):
                if not secrets.compare_digest(self.headers.get('X-StoryForge-Test', ''), app.nonce):
                    self.send(403, {'error': 'Mở lại trang HTML từ ô Colab.'})
                    return
                try:
                    path = self.path.split('?')[0]
                    if post:
                        size = int(self.headers.get('Content-Length', '0'))
                        if not 0 <= size <= (32 * 1024 * 1024 if path == '/api/upload' else 65536):
                            raise ValueError('File tối đa 32 MB; yêu cầu cấu hình tối đa 64 KB.')
                        body = self.rfile.read(size)
                        if path == '/api/upload':
                            if not body.decode('utf-8-sig').strip():
                                raise ValueError('File TXT rỗng.')
                            with app.lock:
                                if app.active:
                                    raise ValueError('Dừng lượt đang chạy trước khi đổi truyện.')
                                app.source.write_bytes(body)
                        elif path == '/api/start':
                            options = json.loads(body)
                            if not isinstance(options, dict):
                                raise ValueError('Cấu hình phải là một object.')
                            app.start(options)
                        elif path == '/api/stop':
                            app.stop()
                        else:
                            self.send(404, {'error': 'Không có endpoint này.'})
                            return
                        self.send(200, {'ok': True})
                    elif path == '/api/status':
                        self.send(200, app.snapshot())
                    elif path.startswith('/api/chunk/'):
                        self.send(200, app.chunk(int(path.rsplit('/', 1)[1])))
                    elif path == '/api/download' and app.merged and not app.active:
                        self.send(200, Path(app.merged).read_bytes(), 'text/plain; charset=utf-8')
                    else:
                        self.send(404, {'error': 'Chưa có kết quả để tải.'})
                except (ValueError, UnicodeError) as error:
                    self.send(400, {'error': str(error)})
                except Exception as error:
                    self.send(500, {'error': type(error).__name__})

        self.server = ThreadingHTTPServer(('127.0.0.1', port), Handler)
        self.port = self.server.server_address[1]
        threading.Thread(target=self.server.serve_forever, daemon=True, name='StoryForge-HTML-server').start()

_SF_STORY_HTML_PAGE = '<!doctype html>\n<html lang="vi">\n<meta charset="utf-8">\n<meta name="viewport" content="width=device-width, initial-scale=1">\n<title>StoryForge · Test dịch 20 luồng</title>\n<style>\n:root{color-scheme:dark;--bg:#12121a;--panel:#191923;--line:#41425e;--text:#f1f5f9;--muted:#b0b9cd;--accent:#a5b4fc}\n*{box-sizing:border-box}body{margin:0;background:#0a0a0f;color:var(--text);font:14px/1.55 system-ui,sans-serif}\nmain{max-width:1440px;margin:auto;padding:28px}h1{font-size:28px;margin:0 0 6px}h2{font-size:18px;margin:18px 0 10px}p{margin:6px 0 14px}.muted,small{color:var(--muted)}\nbutton,input,textarea,select{font:inherit;color:var(--text)}button{min-height:40px;padding:8px 14px;background:#292a43;border:1px solid var(--line);border-radius:6px;cursor:pointer}\nbutton:hover:enabled{background:#373954}button:disabled{opacity:.5;cursor:default}.primary{background:#4f46c7;border-color:#7771dc}.stop{background:#702b39;border-color:#a75b6b}\ninput,textarea,select{background:#222231;border:1px solid var(--line);border-radius:5px;padding:8px;min-width:0}input[type=number]{width:130px}textarea{width:100%;min-height:90px;resize:vertical}\nlabel{display:flex;gap:8px;align-items:center}fieldset{border:0;margin:0;padding:0}fieldset:disabled{opacity:.65}.row{display:flex;flex-wrap:wrap;align-items:center;gap:12px;margin:16px 0}\ndetails{margin:12px 0}summary{cursor:pointer;color:var(--accent);padding:8px 0}#notice{color:var(--accent);min-height:24px}#network{color:#fbbfbc}progress{width:100%;height:12px;accent-color:#818cf8}\n.tabs{display:flex;gap:8px;margin:18px 0 14px;flex-wrap:wrap}.tabs button[aria-selected=true]{background:#343450;border-color:#a5b4fc}\n#lanes{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:12px}.lane{background:var(--panel);border:1px solid var(--line);border-radius:12px;padding:12px;min-width:0}\n.lane header{display:flex;gap:8px;justify-content:space-between;font-weight:600}.badge{color:var(--accent);font-size:12px}.stats{font-size:12px;color:var(--muted);font-variant-numeric:tabular-nums}\n.preview{height:68px;overflow:auto;white-space:pre-wrap;overflow-wrap:anywhere;margin:8px 0}.lane button{min-height:32px;padding:4px 10px}.columns{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:16px}\npre{white-space:pre-wrap;overflow-wrap:anywhere;overflow:auto;max-height:65vh;min-height:260px;padding:16px;background:var(--panel);border:1px solid var(--line);border-radius:8px;font:15px/1.75 system-ui,sans-serif;margin:0}\n#chunk{max-width:100%;flex:1}table{border-collapse:collapse;width:100%;font-variant-numeric:tabular-nums}td,th{padding:9px;text-align:left;border-bottom:1px solid var(--line)}.table-wrap{overflow:auto;max-height:65vh}\n:focus-visible{outline:2px solid #c7d2fe;outline-offset:3px}::selection{background:#5651aa;color:white}*{scrollbar-color:#7b7d99 #222231}input,textarea{caret-color:var(--text)}[hidden]{display:none!important}\n@media(max-width:1100px){#lanes{grid-template-columns:repeat(2,minmax(0,1fr))}}@media(max-width:600px){main{padding:16px}h1{font-size:24px}#lanes,.columns{grid-template-columns:minmax(0,1fr)}input[type=file]{width:100%}.row label{flex-wrap:wrap}td,th{padding:7px}}\n</style>\n<main>\n<h1>Dịch truyện · 20 luồng trực tiếp</h1>\n<p class="muted">Anh Đạt · Bản test HTML dùng máy AI đã nạp. Tải lại trang để xem tiếp lượt đang chạy.</p>\n<fieldset id="settings">\n  <div class="row"><label for="file">Truyện TXT UTF-8 <input id="file" type="file" accept=".txt,text/plain"></label><span id="file-status" class="muted">Chưa chọn file · tối đa 32 MB</span></div>\n  <div class="row">\n    <label>Ký tự/chunk <input id="size" type="number" min="500" max="8000" value="6000"></label>\n    <label>Luồng dịch <input id="workers" type="number" min="1" max="20" value="20"></label>\n    <label>Output token <input id="output" type="number" min="1" value="20000"></label>\n  </div>\n  <details><summary>Prompt và tùy chọn</summary><label for="prompt">Hướng dẫn dịch</label><textarea id="prompt"></textarea><label><input id="retry" type="checkbox">Thử lại chunk lỗi hoặc chạm trần output</label></details>\n  <div class="row"><button id="start" class="primary" type="button">Bắt đầu / tiếp tục</button><button id="preview" type="button">Xem cách cắt chunk</button></div>\n</fieldset>\n<div class="row"><button id="stop" class="stop" disabled>Dừng, giữ tiến trình</button><button id="download" disabled>Tải bản dịch TXT</button><span id="config" class="muted"></span></div>\n<p id="notice" role="status" aria-live="polite">Đang kết nối máy chạy test…</p><p id="network" role="alert"></p>\n<p id="summary" class="stats">Chưa có lượt dịch.</p><progress id="progress" max="1" value="0" aria-label="Tiến độ dịch"></progress>\n<nav class="tabs" aria-label="Xem kết quả"><button data-tab="live" aria-selected="true">20 luồng live</button><button data-tab="detail" aria-selected="false">Nguyên văn và output</button><button data-tab="history" aria-selected="false">Tất cả chunk</button></nav>\n<section id="live"><div id="lanes"></div></section>\n<section id="detail" hidden><label class="row">Xem chunk <select id="chunk"><option value="">Chưa có kế hoạch</option></select></label><div class="columns"><div><h2>Nguyên văn</h2><pre id="source" tabindex="0"></pre></div><div><h2>Bản dịch đang sinh</h2><pre id="translation" tabindex="0"></pre></div></div></section>\n<section id="history" hidden><div class="table-wrap"><table><thead><tr><th>Chunk</th><th>Trạng thái</th><th>Ký tự output</th><th>Lỗi</th></tr></thead><tbody id="history-body"></tbody></table></div></section>\n</main>\n<script>\nconst $=id=>document.getElementById(id), token=\'__SF_NONCE__\';\nlet currentTab=\'live\', busy=false, initialized=false, lastState=null, selectedUpload=null, uploaded=false;\nconst statusName=(status,first)=>({pending:\'Chưa gửi\',running:first==null?\'Chờ token đầu\':\'Đang dịch\',done:\'Hoàn tất\',failed:\'Lỗi\',truncated:\'Chạm trần output\',cancelled:\'Đã dừng\'}[status]||status);\nconst cards=Array.from({length:20},(_,i)=>{\n  const card=document.createElement(\'article\');card.className=\'lane\';\n  card.innerHTML=\'<header><span class="title"></span><span class="badge"></span></header><div class="stats info"></div><div class="preview"></div><div class="stats time"></div><button type="button">Xem đầy đủ</button>\';\n  card.querySelector(\'button\').onclick=()=>{const index=lastState?.lanes[i].index;if(index!=null){$(\'chunk\').value=String(index);showTab(\'detail\');}};\n  $(\'lanes\').append(card);return card;\n});\nfunction text(element,value){if(element.textContent!==String(value))element.textContent=value;}\nasync function api(path,{body,method=\'GET\',raw=false}={}){\n  const response=await fetch(new URL(\'api/\'+path,location.href),{method,headers:{\'X-StoryForge-Test\':token,...(!raw&&body!==undefined?{\'Content-Type\':\'application/json\'}:{})},\n    body:body===undefined?undefined:raw?body:JSON.stringify(body),cache:\'no-store\',signal:AbortSignal.timeout(raw?60000:10000)});\n  if(!response.ok){let message=\'HTTP \'+response.status;try{message=(await response.json()).error||message;}catch{}throw new Error(message);}\n  return response;\n}\nfunction showTab(name){currentTab=name;for(const id of [\'live\',\'detail\',\'history\'])$(id).hidden=id!==name;document.querySelectorAll(\'[data-tab]\').forEach(b=>b.setAttribute(\'aria-selected\',String(b.dataset.tab===name)));if(lastState)render(lastState);if(name===\'detail\')readChunk();}\ndocument.querySelectorAll(\'[data-tab]\').forEach(button=>button.onclick=()=>showTab(button.dataset.tab));\n$(\'chunk\').onchange=readChunk;\n$(\'file\').onchange=()=>{selectedUpload=$(\'file\').files[0];uploaded=false;text($(\'file-status\'),selectedUpload?selectedUpload.name+\' · \'+selectedUpload.size.toLocaleString()+\' byte\':\'Chưa chọn file\');};\nfunction render(state){\n  lastState=state;\n  if(!initialized){for(const id of [\'size\',\'workers\',\'output\',\'prompt\'])$(id).value=state.options[id];$(\'workers\').max=Math.min(20,state.config.parallel||1);$(\'output\').max=state.config.max_tokens||20000;initialized=true;}\n  $(\'settings\').disabled=state.active;$(\'stop\').disabled=!state.active;$(\'download\').disabled=!state.download;\n  if(!selectedUpload)text($(\'file-status\'),state.has_file?\'Đã có file trong phiên Colab · có thể dùng tiếp\':\'Chưa chọn file · tối đa 32 MB\');\n  const labels={idle:\'Anh Đạt, chọn truyện rồi bấm Bắt đầu.\',planning:`Đang đo token: đã đo ${state.measured} chunk, còn ${state.pending} đoạn cần kiểm tra. Chưa gửi dịch.`,running:\'Đang dịch các chunk khác nhau; mỗi thẻ là một luồng.\',stopping:\'Đang đóng request và lưu tiến trình…\',stopped:\'Đã dừng và giữ tiến trình. Bấm Bắt đầu để chạy phần còn lại.\',done:\'Lượt dịch đã kết thúc. Kiểm tra chunk lỗi hoặc chạm trần output trước khi dùng.\',preview:\'Đã chia chunk. Bấm Bắt đầu để dịch.\',error:\'Chưa hoàn tất: \'+state.error};\n  text($(\'notice\'),labels[state.phase]||state.phase);\n  const running=state.chunks.filter(r=>r.status===\'running\').length,waiting=state.chunks.filter(r=>r.status===\'running\'&&r.first_token_seconds==null).length;\n  text($(\'summary\'),`Hoàn tất ${state.completed} / ${state.total} · Đang chạy ${running} · Chờ token đầu ${waiting} · Slot mẫu gần nhất ${state.processing} · Đỉnh ${state.peak} · ${state.seconds}s`);\n  text($(\'config\'),`Máy: ${state.config.parallel||\'?\'} slot · context ${state.config.context||\'?\'} · output tối đa ${state.config.max_tokens||\'?\'}`);\n  $(\'progress\').max=Math.max(1,state.total);$(\'progress\').value=state.completed;\n  state.lanes.forEach((lane,i)=>{const card=cards[i];text(card.querySelector(\'.title\'),`Luồng ${i+1}${lane.index==null?\'\':\' · Chunk \'+(lane.index+1)}`);text(card.querySelector(\'.badge\'),lane.index==null?\'Chưa nhận chunk\':statusName(lane.status,lane.first));\n    text(card.querySelector(\'.info\'),`Input ${lane.input} ký tự · Prompt ${lane.prompt_tokens} token · Output ${lane.characters} ký tự`);\n    text(card.querySelector(\'.preview\'),lane.text||(lane.index==null?\'Bản dịch sẽ xuất hiện ở đây.\':\'Đang chờ nội dung từ model…\'));\n    text(card.querySelector(\'.time\'),`Token đầu: ${lane.first==null?\'chưa nhận\':lane.first+\'s\'} · ${Math.round(lane.seconds)}s${lane.error?\' · \'+lane.error:\'\'}`);card.querySelector(\'button\').disabled=lane.index==null;});\n  if($(\'chunk\').options.length!==state.total||($(\'chunk\').value===\'\'&&state.total)){\n    const previous=$(\'chunk\').value;$(\'chunk\').replaceChildren(...state.chunks.map((r,i)=>new Option(\'Chunk \'+(i+1),String(i))));if(previous&&Number(previous)<state.total)$(\'chunk\').value=previous;\n  }\n  if(currentTab===\'history\'){\n    const fragment=document.createDocumentFragment();state.chunks.forEach(row=>{const tr=document.createElement(\'tr\');for(const value of [row.id,statusName(row.status,row.first_token_seconds),row.characters||0,row.error||\'—\']){const td=document.createElement(\'td\');td.textContent=value;tr.append(td);}fragment.append(tr);});$(\'history-body\').replaceChildren(fragment);\n  }\n}\nasync function readChunk(){if(currentTab!==\'detail\'||!$(\'chunk\').value)return;const index=$(\'chunk\').value;try{const value=await(await api(\'chunk/\'+index)).json();if($(\'chunk\').value===index){text($(\'source\'),value.source);text($(\'translation\'),value.output||\'Chưa có output.\');}}catch(error){text($(\'network\'),error.message);}}\nasync function poll(){if(busy)return;busy=true;try{const state=await(await api(\'status\')).json();render(state);text($(\'network\'),\'\');await readChunk();}catch(error){text($(\'network\'),\'Mất cập nhật: \'+error.message+\' · đang thử kết nối lại.\');}finally{busy=false;}}\nasync function start(preview){\n  $(\'settings\').disabled=true;text($(\'network\'),\'\');text($(\'notice\'),\'Anh Đạt · đã nhận lệnh, đang chuẩn bị file…\');\n  try{\n    if(selectedUpload&&!uploaded){if(selectedUpload.size>32*1024*1024)throw new Error(\'Chọn file TXT tối đa 32 MB.\');await api(\'upload\',{method:\'POST\',body:await selectedUpload.arrayBuffer(),raw:true});uploaded=true;}\n    await api(\'start\',{method:\'POST\',body:{size:Number($(\'size\').value),workers:Number($(\'workers\').value),output:Number($(\'output\').value),prompt:$(\'prompt\').value,preview,retry_failed:$(\'retry\').checked}});\n    await poll();\n  }catch(error){text($(\'network\'),error.message);$(\'settings\').disabled=Boolean(lastState?.active);}\n}\n$(\'start\').onclick=()=>start(false);$(\'preview\').onclick=()=>start(true);\n$(\'stop\').onclick=async()=>{try{await api(\'stop\',{method:\'POST\',body:{}});await poll();}catch(error){text($(\'network\'),error.message);}};\n$(\'download\').onclick=async()=>{try{const blob=await(await api(\'download\')).blob(),url=URL.createObjectURL(blob),a=document.createElement(\'a\');a.href=url;a.download=\'BAN-DICH.txt\';a.click();setTimeout(()=>URL.revokeObjectURL(url),1000);}catch(error){text($(\'network\'),error.message);}};\npoll();setInterval(poll,1000);\n</script>\n</html>\n'

_sf_old_widget_ui = globals().get('SF_STORY_UI')
if _sf_old_widget_ui is not None and _sf_old_widget_ui.task and not _sf_old_widget_ui.task.done():
    print('Anh Đạt · đang dừng lượt test widget cũ và giữ checkpoint…', flush=True)
    _sf_old_widget_ui.stop()
    _, _sf_pending_ui = await asyncio.wait({_sf_old_widget_ui.task}, timeout=10)
    if _sf_pending_ui:
        raise ValueError('Anh Đạt, lượt test cũ chưa dừng xong. Đợi rồi chạy lại ô này; giữ nguyên máy AI.')
if globals().get('SF_STORY_HTML_APP') is None:
    SF_STORY_HTML_APP = StoryHtmlApp(SF_LINKED_STATE, run_colab_story_chunks, _SF_STORY_HTML_PAGE)
    _sf_previous_source = globals().get('SF_CHUNK_SOURCE_FILE')
    if _sf_previous_source and Path(_sf_previous_source).is_file():
        SF_STORY_HTML_APP.source.write_bytes(Path(_sf_previous_source).read_bytes())
    SF_STORY_HTML_APP.serve()
else:
    SF_STORY_HTML_APP.page = _SF_STORY_HTML_PAGE
    if not SF_STORY_HTML_APP.active:
        SF_STORY_HTML_APP.state = SF_LINKED_STATE
        SF_STORY_HTML_APP.runner = run_colab_story_chunks
from google.colab import output as _sf_html_output
print('Anh Đạt · HTML v6 đã sẵn sàng. Mở đường dẫn bên dưới để dùng trang dịch riêng. Không cần nhập key.', flush=True)
_sf_html_output.serve_kernel_port_as_window(SF_STORY_HTML_APP.port)
